# Density Lab: Bias, Variance và Density Estimation

Notebook này giúp bạn quan sát trực tiếp các ý tưởng: density estimation, Gaussian MLE, KDE, bandwidth, bias-variance, GMM và curse of dimensionality.


## 1. Giới thiệu

Ta có dữ liệu mẫu, nhưng thường không biết phân phối thật đã sinh ra dữ liệu. Density estimation là việc ước lượng đường cong mật độ xác suất từ dữ liệu quan sát được.


In [2]:
from pathlib import Path

from density_lab.data_generator import generate_all_datasets
from density_lab.density_models import fit_gaussian_mle
from density_lab.plots import (
    plot_histogram_with_true_density,
    plot_gaussian_mle_fit,
    plot_kde_bandwidth_comparison,
    plot_bias_variance_experiment,
    plot_gmm_components_comparison,
    plot_curse_of_dimensionality,
)

FIGURES = Path("../figures")
FIGURES.mkdir(exist_ok=True)

datasets = generate_all_datasets(n=300, random_state=42)


## 2. Tạo dữ liệu

Ta dùng ba dataset tổng hợp:

- Gaussian: một đỉnh, đối xứng.
- Mixture: hai Gaussian trộn lại, có thể có hai cụm.
- Gamma: phân phối lệch phải.


In [3]:
for name, dataset in datasets.items():
    print(name, dataset["samples"].shape, dataset["description"])
    plot_histogram_with_true_density(dataset, FIGURES / f"notebook_{name}_hist.png")


gaussian (300, 1) Dataset A: chiều cao người, giả sử đến từ một phân phối Gaussian.
mixture (300, 1) Dataset B: hai nhóm con được trộn lại thành một bộ dữ liệu.
gamma (300, 1) Dataset C: dữ liệu lệch phải, giống thời gian chờ hoặc thu nhập.


## 3. Gaussian MLE

Gaussian MLE ước lượng mean và variance từ dữ liệu. Cách này rất tốt nếu dữ liệu thật gần Gaussian, nhưng có bias cao nếu dữ liệu lệch hoặc nhiều cụm.


In [4]:
for name, dataset in datasets.items():
    params = fit_gaussian_mle(dataset["samples"])
    print(name, params)
    plot_gaussian_mle_fit(dataset, FIGURES / f"notebook_{name}_gaussian_mle.png")


gaussian {'mean': 169.71244767198866, 'variance': 42.263531218346465, 'std': 6.501040779624941}
mixture {'mean': 165.64214275068272, 'variance': 78.56423450448946, 'std': 8.863646794885808}
gamma {'mean': 4.034840656418181, 'variance': 7.858440979100042, 'std': 2.803291097817}


## 4. KDE và bandwidth

KDE đặt một bump mượt quanh mỗi điểm dữ liệu. Bandwidth nhỏ làm đường cong bám sát dữ liệu hơn. Bandwidth lớn làm đường cong mượt hơn.


In [5]:
plot_kde_bandwidth_comparison(
    datasets["gaussian"],
    bandwidths=[0.1, 0.3, 0.7, 1.5, 3.0],
    save_path=FIGURES / "notebook_gaussian_kde_bandwidths.png",
)


WindowsPath('../figures/notebook_gaussian_kde_bandwidths.png')

## 5. Bias - variance bằng nhiều train sets

Ta sinh nhiều bộ train khác nhau từ cùng một phân phối thật. Nếu bandwidth nhỏ, KDE thay đổi nhiều giữa các train sets. Nếu bandwidth lớn, đường cong ổn định hơn nhưng dễ quá mượt.


In [6]:
plot_bias_variance_experiment(
    distribution_name="gaussian",
    bandwidths=[0.3, 1.0, 3.0],
    save_path=FIGURES / "notebook_bias_variance_kde.png",
)


WindowsPath('../figures/notebook_bias_variance_kde.png')

## 6. GMM

Gaussian Mixture Model cộng nhiều Gaussian lại với nhau. Với dữ liệu có hai cụm, 2 components thường hợp lý hơn 1 component.


In [7]:
plot_gmm_components_comparison(
    datasets["mixture"],
    components=[1, 2, 5],
    save_path=FIGURES / "notebook_mixture_gmm_components.png",
)


WindowsPath('../figures/notebook_mixture_gmm_components.png')

## 7. Curse of dimensionality

Khi số chiều tăng, không gian lớn rất nhanh. Dữ liệu trở nên thưa, khoảng cách giữa các điểm kém hữu ích hơn, và density estimation trở nên khó hơn.


In [8]:
plot_curse_of_dimensionality(FIGURES / "notebook_curse_of_dimensionality.png")


WindowsPath('../figures/notebook_curse_of_dimensionality.png')

## 8. Kết luận

- Gaussian MLE tốt khi giả định Gaussian đúng.
- KDE linh hoạt nhưng nhạy với bandwidth.
- GMM tốt cho dữ liệu nhiều cụm.
- Dữ liệu nhiều chiều thường cần giảm chiều hoặc domain knowledge trước khi density estimation.
